# 01 - Data loading and exploration

Exploration of the ESM25 metadata (good and bad CV records) before any preprocessing.

Goals:
1. load the metadata of both classes through SeisBench (or from local CSV files in development mode);
2. check which columns are shared and where the label comes from;
3. compare the classes on variables available before the quality review;
4. measure how well each variable alone separates the classes (univariate AUC), to detect possible shortcuts;
5. check whether suspicious associations persist within strata;
6. describe the event and station structure needed for the train/validation/test split;
7. inspect example waveforms with their zero padding (SeisBench mode only).

Label convention: 1 = bad-quality record, 0 = good-quality record.

## 0. Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from wqu.config import ensure_output_dirs, load_config
from wqu.data.exploration import (
    assert_features_shared,
    categorical_distribution_by_label,
    compare_columns,
    constant_columns,
    event_overlap_summary,
    group_size_summary,
    missing_fraction_by_label,
    numeric_summary_by_label,
    stratified_bad_rate,
    univariate_auc_table,
    value_counts_by_label,
)
from wqu.data.loading import (
    load_esm25_metadata,
    load_record_waveform,
    load_seisbench_datasets,
    signal_windows_s,
)
from wqu.visualization import (
    apply_plot_style,
    plot_category_bad_rate,
    plot_numeric_distribution_by_label,
    plot_stratified_bad_rate,
    plot_univariate_auc,
    plot_waveform_with_padding,
    save_figure,
)

CONFIG_PATH = Path("..") / "configs" / "default.yaml"

config = load_config(CONFIG_PATH)
ensure_output_dirs(config.output)
apply_plot_style()
pd.set_option("display.max_rows", 120)

columns = config.columns
exploration = config.exploration
label_names = {subset.label: subset.name for subset in config.data.subsets}
feature_groups = exploration.feature_groups
numeric_features = [feature for group in feature_groups for feature in group.numeric]
categorical_features = [feature for group in feature_groups for feature in group.categorical]

print(f"Data source: {config.data.source}")

## 1. Loading

In [ ]:
metadata_by_subset, metadata = load_esm25_metadata(config)

for name, frame in metadata_by_subset.items():
    print(f"{name:>5}: {len(frame):,} records, {frame.shape[1]} columns")
print(f"total: {len(metadata):,} records")
metadata[columns.label].value_counts().rename(index=label_names)

## 2. Columns and label

Which columns exist in both classes, which are constant, and which column defines the label.
Only shared columns can be compared between classes: a column present in one class only would
separate the classes through its missing values.

In [ ]:
column_presence = compare_columns(metadata_by_subset)
shared_columns = column_presence.index[column_presence["n_subsets"] == len(metadata_by_subset)].tolist()
print(f"Shared columns: {len(shared_columns)}")
for name in metadata_by_subset:
    only_here = column_presence.index[(column_presence["n_subsets"] == 1) & column_presence[name]]
    print(f"Only in {name}: {len(only_here)}")

assert_features_shared(metadata_by_subset, numeric_features + categorical_features)
print("All configured features are shared by both classes.")

In [ ]:
print("Constant shared columns:", constant_columns(metadata, shared_columns))
value_counts_by_label(metadata, "trace_processing_status", columns.label).rename(columns=label_names)

In [ ]:
missing_fraction_by_label(metadata, numeric_features + categorical_features, columns.label).rename(
    columns=label_names
).round(3)

## 3. Distributions by class

Variables available before the quality review, in two groups:
- **physical**: source, path and site properties that legitimately affect signal quality (e.g. signal-to-noise ratio);
- **acquisition**: properties of the recording and of its provenance, which can act as shortcuts for a network trained on waveforms.

In [ ]:
summary = numeric_summary_by_label(metadata, numeric_features, columns.label)
summary.rename(index=label_names, level=columns.label).round(3)

In [ ]:
# (column, logarithmic axis, quantile range shown or None for the full range)
NUMERIC_PLOTS = [
    ("preferred_magnitude", False, None),
    ("path_hyp_dist_km", True, None),
    ("max_unprocessed_pga_cps2", True, (0.01, 0.99)),
    ("recording_span_s", False, None),
    ("original_duration_s", False, None),
    ("component_start_spread_s", False, (0.0, 0.99)),
]

figure, axes = plt.subplots(3, 2, figsize=(11, 10))
for ax, (column, log_scale, quantile_range) in zip(axes.ravel(), NUMERIC_PLOTS):
    plot_numeric_distribution_by_label(
        metadata, column, columns.label, label_names, ax, log_scale=log_scale, quantile_range=quantile_range
    )
figure.tight_layout()
save_figure(figure, config.output.figures_dir / "numeric_distributions_by_label.png")

In [ ]:
overall_bad_rate = float(metadata[columns.label].mean())
CATEGORY_PLOTS = [
    "original_sampling_rate_hz",
    "station_network_code",
    "station_instrument_type_code",
    "original_data_mediator",
    "preferred_magnitude_type",
    "station_preferred_ec8_code",
]

category_tables = {}
figure, axes = plt.subplots(3, 2, figsize=(12, 14))
for ax, column in zip(axes.ravel(), CATEGORY_PLOTS):
    table = categorical_distribution_by_label(metadata, column, columns.label, exploration.top_categories)
    category_tables[column] = table
    plot_category_bad_rate(table, overall_bad_rate, ax)
figure.tight_layout()
save_figure(figure, config.output.figures_dir / "bad_rate_by_category.png")
print(f"Overall bad rate (dashed line): {overall_bad_rate:.3f}")

## 4. Univariate separation of the classes

Each variable is used alone as a score for the bad class.
- Numeric variables are used directly: AUC above 0.5 means larger values indicate bad records.
- Categorical variables are scored with the bad rate of their category, estimated on other folds
  grouped by event (out-of-fold encoding), so that a record is never scored with statistics from its own event.

`separation = max(AUC, 1 - AUC)`: 0.5 means no information, 1 means perfect separation.
High separation in the physical group is expected; high separation in the acquisition group signals
a possible shortcut.

In [ ]:
auc_table = univariate_auc_table(
    metadata,
    feature_groups,
    label_column=columns.label,
    group_column=columns.event_id,
    n_splits=exploration.n_splits,
    smoothing=exploration.target_encoding_smoothing,
    seed=exploration.random_seed,
)
auc_table.to_csv(config.output.tables_dir / "univariate_auc.csv", index=False)
auc_table.round(3)

In [ ]:
figure = plot_univariate_auc(auc_table, [group.name for group in feature_groups])
save_figure(figure, config.output.figures_dir / "univariate_auc.png")

## 5. Stratified bad rates

Does the association of a variable with the label persist within each level of a stratum?
If, for example, the bad rate depends on the original sampling rate only because some networks
use certain rates, the dependence disappears within each network.
Cells with fewer than `min_stratum_count` records are left empty.

In [ ]:
for comparison in exploration.stratified_comparisons:
    rates, counts = stratified_bad_rate(
        metadata,
        variable=comparison.variable,
        stratum=comparison.stratum,
        label_column=columns.label,
        n_bins=exploration.n_quantile_bins,
        top_n=exploration.top_categories,
        max_discrete_levels=exploration.max_discrete_levels,
        min_count=exploration.min_stratum_count,
    )
    figure, ax = plt.subplots(figsize=(1.0 * rates.shape[1] + 3, 0.5 * rates.shape[0] + 2))
    plot_stratified_bad_rate(rates, counts, ax)
    ax.set_title(f"bad rate: {comparison.variable} within {comparison.stratum}", loc="left")
    figure.tight_layout()
    save_figure(figure, config.output.figures_dir / f"stratified_{comparison.variable}_by_{comparison.stratum}.png")
    plt.show()

## 6. Structure for the data split

Records of the same event (and possibly of the same station) must not be split between training and test.
These tables show how many events contain both classes and how large the groups are.

In [ ]:
event_overlap_summary(metadata, columns.event_id, columns.label)

In [ ]:
group_size_summary(metadata, [columns.event_id, columns.station_id, "origin_year"]).round(1)

## 7. Waveform inspection

Requires `data.source: seisbench` in the configuration (waveforms downloaded into the SeisBench cache).
Shaded intervals are zero padding: front padding aligns the three components, back padding fills the trace to 420 s.

In [ ]:
if config.data.source == "seisbench":
    datasets = load_seisbench_datasets(config)
    sampling_rate_hz = float(metadata["trace_sampling_rate_hz"].iloc[0])
    examples = metadata.groupby(columns.label, group_keys=False).sample(
        n=exploration.n_waveform_examples, random_state=exploration.random_seed
    )
    for _, record in examples.iterrows():
        waveform = load_record_waveform(record, datasets, columns)
        title = f"{label_names[record[columns.label]]}: {record['trace_name_old']}"
        figure = plot_waveform_with_padding(waveform, sampling_rate_hz, signal_windows_s(record), title)
        save_figure(figure, config.output.figures_dir / f"waveform_{record['trace_name_old']}.png")
        plt.show()
else:
    print("Waveform inspection skipped: set data.source to 'seisbench' once the datasets are downloaded.")